### Using this notebook for hyperparameter tuning

In [10]:
#importing libraries

import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sklearn.model_selection import GridSearchCV



In [2]:
#importing dataset into dataframe

data = pd.read_csv("churn_modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#data preprocessing
data.drop(columns=['RowNumber', 'Surname', 'CustomerId'], inplace=True)

#splitting data into x and y
x = data.drop(columns=['Exited'])
y = data['Exited']

#splitting data in train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

#encoding categorical features
label_encoder = LabelEncoder()
x_train['Gender'] = label_encoder.fit_transform(x_train['Gender'])
x_test['Gender'] = label_encoder.transform(x_test['Gender'])

onehot_encoder = OneHotEncoder()
geo_train_arr = onehot_encoder.fit_transform(x_train[['Geography']]).toarray()
geo_test_arr = onehot_encoder.transform(x_test[['Geography']]).toarray()

geo_train_df = pd.DataFrame(geo_train_arr, columns=onehot_encoder.get_feature_names_out())
geo_test_df = pd.DataFrame(geo_test_arr, columns=onehot_encoder.get_feature_names_out())

x_train = pd.concat([x_train.reset_index(drop=True), geo_train_df], axis=1)
x_test = pd.concat([x_test.reset_index(drop=True), geo_test_df], axis=1)

x_train.drop(columns=['Geography'], inplace=True)
x_test.drop(columns=['Geography'], inplace=True)

#scaling data features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [6]:
#saving ojects to pickle files

with open("onehot_encoder_hp.pkl", "wb") as file_obj:
    pickle.dump(onehot_encoder,file_obj)

with open("scaler_hp.pkl", "wb") as file_obj:
    pickle.dump(scaler, file_obj)

with open("label_encoder_hp.pkl", "wb") as file_obj:
    pickle.dump(label_encoder, file_obj)
        

In [8]:
#custom function to create model

def create_model(neurons:int, layers:int):
    "this function creates model based on neurons and layers passed"

    model = Sequential([
        Dense(units=neurons, activation="relu", input_shape=(x_train.shape[1], ))
        
    ])

    for _ in range(layers-1):
        model.add(Dense(units=neurons, activation="relu"))

    model.add(Dense(units=1, activation="sigmoid"))

    return model    

In [ ]:
model = KerasClassifier(build_fn=create_model,neurons=32, loss="binary_crossentropy", layers=1, epochs=50, batch_size=10, verbose=True)

params_grid = {
    "neurons":[16, 32, 64, 128],
    "epochs": [50, 100],
    "layers": [1, 2]
}


grid_search = GridSearchCV(estimator=model, cv=3, param_grid=params_grid, verbose=1)
grid_result = grid_search.fit(x_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-p

ValueError: 
All the 48 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py", line 1501, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py", line 770, in fit
   